# Basics

## WER and CER

Word Error rate.  
Character Error Rate.  

When we run a Speech to Text model, we need a metric to define how well the model performed.  
We can't just say yes or no, we need to give metrics which can give us the percentage of it.  

### WER

There are 3 kinds of errors:

1. Substituition - S: When we need to replace a word with another
2. Insertion - I: When we need to add a word somewhere
3. Deletion - D: To delete a word

In WER, it adds up all the errors, and then divides it by the length of the orignal word.  

wer = (S+I+D)/N

This WER gives us the percentage of how accurate the model was. 

### CER

Sometimes, words are mostly right, but there may be a tiny error in spelling. For example cat instead of cats.  

WER would tell us the word is completely wrong, but it is mostly correct, so we need CER over here as a metric. 

CER gives us the character by character error.  

CER is useful for languages like Chinese, Japanese etc which dont have spaces in their languaes. 

CER is also useful for Indic languaes. If a hindi word only has a matra error, it is not fully incorrect, only partially.

### Jewer Library

To find WER and CER, we use the Jewer library. 

In [2]:
from jiwer import wer, cer

reference  = "the cat sat on the mat"
prediction = "the bat sat on mat"

print(wer(reference, prediction))   # 0.333...
print(cer(reference, prediction))   # character-level score

0.3333333333333333
0.22727272727272727


We can see that CER performs better here as the wrong words are pretty similar to the original ones.

In [ ]:
# we can also pass a batch of sentences with a list

references  = ["hello world", "i love samosas", "good morning"]
predictions = ["hello word",  "i love samosa", "good morning"]

print(wer(references, predictions))   # averaged over the batch

0.2857142857142857


In [5]:
from jiwer import process_words

out = process_words(reference, prediction)
print(out)
# Shows substitutions, insertions, deletions, hits separately

WordOutput(references=[['the', 'cat', 'sat', 'on', 'the', 'mat']], hypotheses=[['the', 'bat', 'sat', 'on', 'mat']], alignments=[[AlignmentChunk(type='equal', ref_start_idx=0, ref_end_idx=1, hyp_start_idx=0, hyp_end_idx=1), AlignmentChunk(type='substitute', ref_start_idx=1, ref_end_idx=2, hyp_start_idx=1, hyp_end_idx=2), AlignmentChunk(type='equal', ref_start_idx=2, ref_end_idx=4, hyp_start_idx=2, hyp_end_idx=4), AlignmentChunk(type='delete', ref_start_idx=4, ref_end_idx=5, hyp_start_idx=4, hyp_end_idx=4), AlignmentChunk(type='equal', ref_start_idx=5, ref_end_idx=6, hyp_start_idx=4, hyp_end_idx=5)]], wer=0.3333333333333333, mer=0.3333333333333333, wil=0.4666666666666667, wip=0.5333333333333333, hits=4, substitutions=1, insertions=0, deletions=1)


## Code Mixing

When we use more than 1 language in a sentencee, its called code mixing.  

eg. "I told him ki kal aana", "Whatsapp kar do"

These sentences break on models which were trained on hindi and english data seperately.

### TOkenizer Problem
Usually a tokenizer will chop sentences into words which it knows. But if it doesnt know the word, it will keep on chopping it to the byte level. So for many Indic language words, it may just keep chopping them into plain characters. 

This doesn't work for a few reasons:

1. More tokenizers means more processing, slows down the model.
2. If you break down a whole word into its characters, it loses meaning. 
3. Worse predictions. The model first had to guess one correct word, but now it will have to guess 6 of them correctly.

## Beam Search vs Greedy Decoding

### Greedy
In a GPT, when predicting the next word, there are a lot of options with the transformer. Each option has a probability distribution along with it. The word with the highest probability is chosen. 

### BEam Search
Instead of only taking the most probable word, we take the top k probable words. k is user defined, and we can choose it. It takes the top k tokens and then for the next, it takes the top 3 again. At the end, it takes the sequence of words with the highest probability.  
Pros: This is useful when a less probable word leads to a better one after.
Cons: Slower, more memory used. 